In [ ]:
from google.colab import files
uploaded = files.upload()


Saving titanic.zip to titanic (1).zip


In [ ]:
import zipfile, os

zip_path = list(uploaded.keys())[0]
extract_path = "titanic_data"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Extracted files:", os.listdir(extract_path))


Extracted files: ['gender_submission.csv', 'test.csv', 'train.csv']


In [ ]:
import pandas as pd

train = pd.read_csv(os.path.join(extract_path, 'train.csv'))
test  = pd.read_csv(os.path.join(extract_path, 'test.csv'))

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


Train shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
print("Train columns and types:")
print(train.dtypes)
print("\nMissing values (train):\n", train.isnull().sum())
display(train.describe())


Train columns and types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Missing values (train):
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [ ]:

import numpy as np

def feature_engineer(df):
    df = df.copy()

    df['Title'] = df['Name'].str.extract(r',\s*([^.]*)\.', expand=False)
    df['Title'] = df['Title'].replace({
        'Mlle':'Miss','Ms':'Miss','Mme':'Mrs',
        'Lady':'Rare','Countess':'Rare','Capt':'Rare','Col':'Rare','Don':'Rare',
        'Dr':'Rare','Major':'Rare','Rev':'Rare','Sir':'Rare','Jonkheer':'Rare','Dona':'Rare'
    })


    df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
    df['Age'].fillna(df['Age'].median(), inplace=True)


    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
    df['Fare'].fillna(df['Fare'].median(), inplace=True)


    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)


    df['Sex'] = df['Sex'].map({'male':0,'female':1}).astype(int)


    df['AgeBand'] = pd.cut(df['Age'], bins=[0,12,20,40,60,120], labels=[0,1,2,3,4]).astype(int)
    df['FareBand'] = pd.qcut(df['Fare'], 4, labels=False)


    df.drop(['Cabin','Ticket','Name'], axis=1, inplace=True, errors='ignore')


    df = pd.get_dummies(df, columns=['Embarked','Title'], drop_first=True)
    return df


train_fe = feature_engineer(train)
test_fe  = feature_engineer(test)

print("train_fe shape:", train_fe.shape)
print("test_fe shape:", test_fe.shape)
train_fe.head()


train_fe shape: (891, 19)
test_fe shape: (418, 17)


/tmp/ipython-input-2140947302.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipython-input-2140947302.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,IsAlone,AgeBand,FareBand,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Title_the Countess
0,1,0,3,0,22.0,1,0,7.2500,2,0,2,0,False,True,False,True,False,False,False
1,2,1,1,1,38.0,1,0,71.2833,2,0,2,3,False,False,False,False,True,False,False
2,3,1,3,1,26.0,0,0,7.9250,1,1,2,1,False,True,True,False,False,False,False
3,4,1,1,1,35.0,1,0,53.1000,2,0,2,3,False,True,False,False,True,False,False
4,5,0,3,0,35.0,0,0,8.0500,1,1,2,1,False,True,False,True,False,False,False


In [ ]:
X = train_fe.drop('Survived', axis=1)
y = train_fe['Survived']
X_test = test_fe.reindex(columns=X.columns, fill_value=0)

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)


X shape: (891, 18)
X_test shape: (418, 18)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2,
                                                  random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_val_s)
print("Validation accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))


Validation accuracy: 0.8547486033519553
              precision    recall  f1-score   support

           0       0.86      0.91      0.88       110
           1       0.84      0.77      0.80        69

    accuracy                           0.85       179
   macro avg       0.85      0.84      0.84       179
weighted avg       0.85      0.85      0.85       179



In [ ]:
test_preds = model.predict(X_test_s).astype(int)
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': test_preds})
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv")

from google.colab import files
files.download('submission.csv')


Saved submission.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>